<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/13_mechanistic_qat_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import json
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# 1. Environment & Paths
MODEL_PATH = "/content/drive/MyDrive/ResearchProject/phi3-bucket-collapse/models/target_model_fp16"
INPUT_JSON_PATH = "/content/drive/MyDrive/ResearchProject/trace_map.json"
MIN_CLUSTER_SIZE = 16  # Minimum batching threshold to prevent layer overfitting
BATCH_SIZE = 4         # Safe batch size for Colab A100

# 2. PyTorch Dataset Class
class MUSE_Dataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.texts = texts
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encodings = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            'input_ids': encodings['input_ids'].squeeze(),
            'attention_mask': encodings['attention_mask'].squeeze()
        }

# 3. Load Tokenizer
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4. Load and Route the JSON Data
print("\n--- Initializing Data Router ---")
with open(INPUT_JSON_PATH, "r") as f:
    trace_map = json.load(f)

# Initialize raw cluster bins (32 layers)
raw_clusters = {layer: [] for layer in range(32)}

for article_id, data in trace_map.items():
    # Strategy 3 groups by the PRIMARY (index 0) layer responsible for the fact
    primary_layer = data['top_layers'][0]
    raw_clusters[primary_layer].append(data['text'])

# 5. Filter and Build DataLoaders
cluster_dataloaders = {}
residual_texts = []

print("\n📊 Cluster Distribution Analysis:")
print("-" * 40)
print(f"{'Layer ID':<10} | {'Total Articles':<15} | {'Status'}")
print("-" * 40)

for layer, texts in raw_clusters.items():
    count = len(texts)

    if count == 0:
        continue # Skip empty layers entirely

    if count >= MIN_CLUSTER_SIZE:
        print(f"Layer {layer:<5} | {count:<15} | ✅ Validated for Batched QAT")
        dataset = MUSE_Dataset(texts, tokenizer)
        # We shuffle to ensure batches within the cluster are randomized
        cluster_dataloaders[layer] = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    else:
        print(f"Layer {layer:<5} | {count:<15} | ⚠️ Too small. Moved to Residuals.")
        residual_texts.extend(texts)

# 6. Handle the Residual Dataset
if len(residual_texts) > 0:
    print("-" * 40)
    print(f"Residuals  | {len(residual_texts):<15} | 🔄 Grouped into catch-all loader")
    residual_dataset = MUSE_Dataset(residual_texts, tokenizer)
    residual_dataloader = DataLoader(residual_dataset, batch_size=BATCH_SIZE, shuffle=True)
else:
    residual_dataloader = None

print("\n✅ Data Routing Complete!")
print(f"Total Valid Layer Clusters: {len(cluster_dataloaders)}")

Loading Tokenizer...

--- Initializing Data Router ---

📊 Cluster Distribution Analysis:
----------------------------------------
Layer ID   | Total Articles  | Status
----------------------------------------
Layer 0     | 92              | ✅ Validated for Batched QAT
Layer 1     | 176             | ✅ Validated for Batched QAT
Layer 2     | 143             | ✅ Validated for Batched QAT
Layer 3     | 16              | ✅ Validated for Batched QAT
Layer 4     | 20              | ✅ Validated for Batched QAT
Layer 5     | 18              | ✅ Validated for Batched QAT
Layer 6     | 59              | ✅ Validated for Batched QAT
Layer 7     | 37              | ✅ Validated for Batched QAT
Layer 8     | 18              | ✅ Validated for Batched QAT
Layer 9     | 10              | ⚠️ Too small. Moved to Residuals.
Layer 10    | 14              | ⚠️ Too small. Moved to Residuals.
Layer 11    | 9               | ⚠️ Too small. Moved to Residuals.
Layer 12    | 5               | ⚠️ Too small. Moved t

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import copy
import gc
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Define Utility-Preserving Unlearning Loss
class UtilityPreservingUnlearningLoss(nn.Module):
    def __init__(self, grid_margin=0.133, lambda_reg=50.0, alpha_retain=50.0):
        super().__init__()
        self.grid_margin = grid_margin
        self.lambda_reg = lambda_reg
        self.alpha_retain = alpha_retain

    def forward(self, forget_logits, forget_labels, retain_logits, retain_labels, current_weights, original_weights):
        # Forget Loss (Gradient Ascent)
        shift_f_logits = forget_logits[..., :-1, :].contiguous().float()
        shift_f_labels = forget_labels[..., 1:].contiguous()
        f_ce_loss = F.cross_entropy(shift_f_logits.view(-1, shift_f_logits.size(-1)), shift_f_labels.view(-1))
        forget_loss = -torch.clamp(f_ce_loss, max=50.0)

        # Retain Loss (Language Modeling)
        shift_r_logits = retain_logits[..., :-1, :].contiguous().float()
        shift_r_labels = retain_labels[..., 1:].contiguous()
        retain_loss = F.cross_entropy(shift_r_logits.view(-1, shift_r_logits.size(-1)), shift_r_labels.view(-1), ignore_index=tokenizer.pad_token_id)

        # Grid Penalty (Quantization Margin)
        weight_diff = torch.abs(current_weights - original_weights)
        grid_penalty = torch.relu(self.grid_margin - weight_diff).mean()

        total_loss = forget_loss + (self.alpha_retain * retain_loss) + (self.lambda_reg * grid_penalty)
        return total_loss, forget_loss, retain_loss, grid_penalty

# 2. Setup Training Environment
print("\nLoading Retain DataLoader for Utility Anchoring...")
RETAIN_SET_PATH = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/retain_set.csv"
retain_df = pd.read_csv(RETAIN_SET_PATH)
retain_texts = retain_df['text'].fillna("").astype(str).tolist()
retain_dataset = MUSE_Dataset(retain_texts, tokenizer)
retain_dataloader = DataLoader(retain_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("\nLoading Base Phi-3 Model to GPU...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    torch_dtype=torch.float16,
    device_map={"": 0}
)

criterion = UtilityPreservingUnlearningLoss()
scaler = torch.amp.GradScaler('cuda')

def get_flat_weights(modules):
    return torch.cat([p.view(-1) for m in modules for p in m.parameters()])

# 3. Sort Clusters Top-Down (Layer 31 down to Layer 0)
# We process residuals at the very end using a generalized late-stage block
sorted_layers = sorted(cluster_dataloaders.keys(), reverse=True)
retain_iter = iter(retain_dataloader)

print("\n--- Starting Top-Down Clustered Unlearning ---")

for target_layer in sorted_layers:
    current_dataloader = cluster_dataloaders[target_layer]
    num_batches = len(current_dataloader)

    print(f"\n[+] Processing Layer {target_layer} Cluster | {num_batches} batches")

    # A. Freeze entire model
    for param in model.parameters():
        param.requires_grad = False

    # B. Surgically unfreeze ONLY the targeted MLP layer
    mlp_module = model.model.layers[target_layer].mlp
    mlp_module.to(torch.float32) # Upcast for stability
    for param in mlp_module.parameters():
        param.requires_grad = True

    target_modules = [mlp_module]
    original_block_weights = get_flat_weights(target_modules).clone().detach()

    # Optimizer for this specific layer
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

    model.train()
    epochs = 5  # Standard for batched QAT
    best_grid_penalty = float('inf')
    best_weights = None

    # C. Batched QAT Loop
    for epoch in range(epochs):
        epoch_loss = 0.0

        for step, forget_batch in enumerate(current_dataloader):
            optimizer.zero_grad()

            f_inputs = forget_batch['input_ids'].to("cuda")
            f_attn = forget_batch['attention_mask'].to("cuda")
            f_labels = f_inputs.clone()

            # Pull healthy retain batch
            try:
                retain_batch = next(retain_iter)
            except StopIteration:
                retain_iter = iter(retain_dataloader)
                retain_batch = next(retain_iter)

            r_inputs = retain_batch['input_ids'].to("cuda")
            r_attn = retain_batch['attention_mask'].to("cuda")
            r_labels = r_inputs.clone()

            with torch.autocast("cuda", dtype=torch.float16):
                f_outputs = model(input_ids=f_inputs, attention_mask=f_attn)
                r_outputs = model(input_ids=r_inputs, attention_mask=r_attn)

                current_block_weights = get_flat_weights(target_modules)

                loss, _, _, g_penalty = criterion(
                    f_outputs.logits, f_labels,
                    r_outputs.logits, r_labels,
                    current_block_weights, original_block_weights
                )

            if math.isnan(loss.item()):
                print(f"⚠️ NaN at Epoch {epoch+1}, Step {step}. Stopping layer.")
                break

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()

            if g_penalty.item() < best_grid_penalty:
                best_grid_penalty = g_penalty.item()
                best_weights = [copy.deepcopy(m.state_dict()) for m in target_modules]

        print(f"  -> Epoch {epoch+1}/{epochs} | Avg Loss: {epoch_loss/num_batches:.2f} | Final Penalty: {g_penalty.item():.4f}")

    # Restore safest quantized state for this layer before moving down
    if best_weights is not None:
        for idx, m in enumerate(target_modules):
            m.load_state_dict(best_weights[idx])

    # Cleanup memory before next layer
    mlp_module.to(torch.float16)
    torch.cuda.empty_cache()
    gc.collect()

# 4. Handle Residuals (Optional fallback to Layer 31)
if residual_dataloader is not None:
    print(f"\n[+] Processing Residuals | {len(residual_dataloader)} batches on default Layer 31")
    # You can apply a similar unfreezing and training loop here targeting Layer 31
    # as a generalized fallback for the random outlier facts.

# 5. Save the Final Model
OUTPUT_MODEL_DIR = "/content/drive/MyDrive/ResearchProject/models/QSurgical_Clustered_FP16"
model.save_pretrained(OUTPUT_MODEL_DIR)
tokenizer.save_pretrained(OUTPUT_MODEL_DIR)
print(f"\n✅ Full Layer-Clustered Q-Surgical Framework Complete! Model saved to {OUTPUT_MODEL_DIR}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!



Loading Retain DataLoader for Utility Anchoring...

Loading Base Phi-3 Model to GPU...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]


--- Starting Top-Down Clustered Unlearning ---

[+] Processing Layer 31 Cluster | 18 batches
  -> Epoch 1/5 | Avg Loss: 102.34 | Final Penalty: 0.1326
  -> Epoch 2/5 | Avg Loss: 98.96 | Final Penalty: 0.1325
  -> Epoch 3/5 | Avg Loss: 100.40 | Final Penalty: 0.1324
  -> Epoch 4/5 | Avg Loss: 101.86 | Final Penalty: 0.1323
  -> Epoch 5/5 | Avg Loss: 104.29 | Final Penalty: 0.1322

[+] Processing Layer 30 Cluster | 6 batches
  -> Epoch 1/5 | Avg Loss: 99.23 | Final Penalty: 0.1328
  -> Epoch 2/5 | Avg Loss: 108.21 | Final Penalty: 0.1327
  -> Epoch 3/5 | Avg Loss: 101.92 | Final Penalty: 0.1326
  -> Epoch 4/5 | Avg Loss: 104.33 | Final Penalty: 0.1325
  -> Epoch 5/5 | Avg Loss: 109.34 | Final Penalty: 0.1325

[+] Processing Layer 29 Cluster | 7 batches
  -> Epoch 1/5 | Avg Loss: 96.55 | Final Penalty: 0.1328
  -> Epoch 2/5 | Avg Loss: 99.91 | Final Penalty: 0.1327
  -> Epoch 3/5 | Avg Loss: 99.16 | Final Penalty: 0.1326
  -> Epoch 4/5 | Avg Loss: 105.56 | Final Penalty: 0.1325
  -> Epoc

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Full Layer-Clustered Q-Surgical Framework Complete! Model saved to /content/drive/MyDrive/ResearchProject/models/QSurgical_Clustered_FP16
